# Kaggle V2: İleri Seviye Optimizasyon ve Pipeline
Bu notebook, veri setinin temizlenmiş ve onaylanmış temel özellik mühendisliği (Feature Engineering) adımlarını içerir. Buradan itibaren tüm yeni özellikler ve dönüşümler izole bir şekilde test edilecektir.

## 1. Temel Veri Hazırlığı (Baseline Pipeline)
* Eksik veriler medyan ve mod ile dolduruldu.
* Bilişsel Yük Endeksi ve Uyku Hijyeni gibi güçlü sinyaller eklendi.
* Kategorik değişkenler sayısallaştırıldı (OHE).
* Algoritmanın önem vermediği gürültü sütunlar matristen budandı.

In [28]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- 1. VERİ YÜKLEME ---
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

# --- 2. EKSİK VERİ DOLDURMA ---
sayisal_kolonlar = train.select_dtypes(include=['float64', 'int64']).columns
sayisal_kolonlar = sayisal_kolonlar.drop(['id', 'bilissel_performans_skoru'])
kategorik_kolonlar = train.select_dtypes(include=['object']).columns

for col in sayisal_kolonlar:
    medyan = train[col].median()
    train[col] = train[col].fillna(medyan)
    test[col] = test[col].fillna(medyan)

for col in kategorik_kolonlar:
    mod = train[col].mode()[0]
    train[col] = train[col].fillna(mod)
    test[col] = test[col].fillna(mod)

# --- 3. KATEGORİK KODLAMA (One-Hot Encoding) ---
train_encoded = pd.get_dummies(train, columns=kategorik_kolonlar, drop_first=True)
test_encoded = pd.get_dummies(test, columns=kategorik_kolonlar, drop_first=True)
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

if 'bilissel_performans_skoru' in test_encoded.columns:
    test_encoded = test_encoded.drop(columns=['bilissel_performans_skoru'])

# --- 4. ÖZELLİK MÜHENDİSLİĞİ (Feature Engineering) ---
def ozellik_muhendisligi(df):
    df = df.copy()
    df['is_extreme_caffeine'] = (df['uyku_oncesi_kafein_mg'] > 300).astype(int)
    df['is_obese'] = (df['vucut_kitle_indeksi'] > 30).astype(int)
    df['dinlendirici_uyku_orani'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['bilissel_yuk_endeksi'] = df['stres_skoru'] * df['gunluk_calisma_saati']
    df['uyku_hijyeni_ihlali'] = df['uyku_oncesi_kafein_mg'] * df['uyku_oncesi_ekran_suresi_dk']
    df['gece_huzursuzlugu'] = df['uykuya_dalma_suresi_dk'] * df['gecelik_uyanma_sayisi']
    return df

train_fe = ozellik_muhendisligi(train_encoded)
test_fe = ozellik_muhendisligi(test_encoded)

# --- 5. GÜRÜLTÜ SÜTUNLARIN BUDANMASI ---
etkisiz_elemanlar = ['kronotip_Sabah insani', 'ulke_Sweden', 'ulke_Fransa', 'ulke_Arjantin']
train_fe = train_fe.drop(columns=etkisiz_elemanlar, errors='ignore')
test_fe = test_fe.drop(columns=etkisiz_elemanlar, errors='ignore')

# --- 6. X / y AYRIMI ---
X = train_fe.drop(columns=['id', 'bilissel_performans_skoru'])
y = train_fe['bilissel_performans_skoru']
test_id = test_fe['id']
X_test = test_fe.drop(columns=['id'])

print(f'Pipeline tamamlandi!')
print(f'X: {X.shape} | y: {y.shape} | X_test: {X_test.shape}')

Pipeline tamamlandi!
X: (56000, 49) | y: (56000,) | X_test: (24000, 49)


---
## 2. Deney 1: Hedef Değişken (Target) Manipülasyonu
**Hipotez:** RMSE metriği uç değerleri (büyük hataları) aşırı cezalandırır. Hedef değişkenin (Bilişsel Performans Skoru) logaritmasını alarak dağılımı normalleştirirsek, ağaç modeli varyansı daha iyi kavrayabilir ve hata payımız düşer. 

**Metodoloji:**
1. Gerçek hedef değişken kopyalanır.
2. Hedefin `log1p` (1 eklenmiş logaritma) değeri hesaplanır.
3. Model logaritmik uzayda eğitilir.
4. Çıkan tahmin sonuçları, gerçek hatayı görebilmek için `expm1` (üstel fonksiyon) ile tekrar gerçek uzaya dönüştürülüp ölçülür.

In [29]:
from sklearn.model_selection import KFold

# Gerçek y değerini yedekleyelim
y_orijinal = y.copy()

# Hedefin logaritmasını alalım (log1p kullanımı, veride 0 değeri varsa sonsuza gitmesini engeller)
y_log = np.log1p(y_orijinal)

# 5 Katlı test ortamını kuralım
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_orijinal = []
rmse_log = []

for train_index, val_index in kf.split(X):
    X_train_f, X_val_f = X.iloc[train_index], X.iloc[val_index]
    
    # --- SENARYO A: ORİJİNAL HEDEF İLE EĞİTİM ---
    y_train_orij, y_val_orij = y_orijinal.iloc[train_index], y_orijinal.iloc[val_index]
    model_orij = LGBMRegressor(random_state=42, n_jobs=-1)
    model_orij.fit(X_train_f, y_train_orij)
    tahmin_orij = model_orij.predict(X_val_f)
    rmse_orijinal.append(np.sqrt(mean_squared_error(y_val_orij, tahmin_orij)))
    
    # --- SENARYO B: LOGARİTMİK HEDEF İLE EĞİTİM ---
    y_train_log, y_val_log = y_log.iloc[train_index], y_log.iloc[val_index]
    model_log = LGBMRegressor(random_state=42, n_jobs=-1)
    model_log.fit(X_train_f, y_train_log)
    
    # Tahminler şu an logaritmik uzayda (Örn: 2.1). Bunu gerçek skora çevirmeliyiz
    tahmin_log_uzayi = model_log.predict(X_val_f)
    tahmin_geri_donusum = np.expm1(tahmin_log_uzayi) # Üstel dönüşüm
    
    # Hatayı, tahminin dönüştürülmüş hali ile gerçek y_val_orij arasında ölçüyoruz
    rmse_log.append(np.sqrt(mean_squared_error(y_val_orij, tahmin_geri_donusum)))

print("-" * 40)
print(f"Orijinal Hedef ile Ortalama RMSE   : {np.mean(rmse_orijinal):.4f}")
print(f"Logaritmik Hedef ile Ortalama RMSE : {np.mean(rmse_log):.4f}")

fark = np.mean(rmse_orijinal) - np.mean(rmse_log)
if fark > 0:
    print(f"\nSONUÇ: Başarılı! Log dönüşümü skoru {fark:.5f} iyileştirdi. Artık 'y_log' üzerinden ilerliyoruz.")
    y = y_log # Pipeline'ın kalanı için y'yi güncelliyoruz
else:
    print(f"\nSONUÇ: Başarısız. Log dönüşümü işe yaramadı. Orijinal 'y' ile devam ediyoruz.")

----------------------------------------
Orijinal Hedef ile Ortalama RMSE   : 1.2341
Logaritmik Hedef ile Ortalama RMSE : 1.2496

SONUÇ: Başarısız. Log dönüşümü işe yaramadı. Orijinal 'y' ile devam ediyoruz.


---
## 3. Deney 2: Target Encoding (Meslek ve Ülke)
**Hipotez:** Yüksek kardinaliteye (farklı çeşit sayısına) sahip kategorik değişkenlerin hedef değişken ortalamalarını hesaplayarak modele yeni ve güçlü sayısal sinyaller vermek. Veri sızıntısını (Leakage) önlemek için hesaplamalar Fold döngüsü içinde yapılacaktır.

In [30]:
# Hangi sütunlara Target Encoding uygulayacağımızı seçiyoruz
# Bu veriler OHE yapılmamış, ham train/test tablolarımızda duruyor
hedef_kodlanacaklar = ['meslek', 'ulke']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_eski_matris = []
rmse_te_matris = []

for train_index, val_index in kf.split(X):
    # Orijinal X matrisimiz (OHE yapılmış hali)
    X_train_eski, X_val_eski = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]
    
    # 1. Eski matrisle eğitim (Referans noktamız)
    model_eski = LGBMRegressor(random_state=42, n_jobs=-1)
    model_eski.fit(X_train_eski, y_train_fold)
    tahmin_eski = model_eski.predict(X_val_eski)
    rmse_eski_matris.append(np.sqrt(mean_squared_error(y_val_fold, tahmin_eski)))
    
    # 2. Target Encoding'li yeni matrisi hazırlama
    # Kopyalama yapıyoruz ki ana matrisi bozmayalım
    X_train_yeni = X_train_eski.copy()
    X_val_yeni = X_val_eski.copy()
    
    # Ham 'train' verisinden sadece bu fold'daki indeksleri alalım
    ham_train_fold = train.iloc[train_index]
    ham_val_fold = train.iloc[val_index]
    
    for sutun in hedef_kodlanacaklar:
        # Sadece EĞİTİM setindeki grupların hedef ortalamasını alıyoruz
        ortalama_sozluk = y_train_fold.groupby(ham_train_fold[sutun]).mean()
        
        # Öğrenilen bu ortalamaları yeni kolon olarak ekliyoruz
        X_train_yeni[f'{sutun}_target_enc'] = ham_train_fold[sutun].map(ortalama_sozluk)
        X_val_yeni[f'{sutun}_target_enc'] = ham_val_fold[sutun].map(ortalama_sozluk)
        
        # Eğer validasyon setinde eğitimde hiç görmediğimiz bir meslek/ülke gelirse (NaN olursa), 
        # genel ortalama ile dolduruyoruz ki kod çökmesin
        genel_ortalama = y_train_fold.mean()
        X_val_yeni[f'{sutun}_target_enc'].fillna(genel_ortalama, inplace=True)
        X_train_yeni[f'{sutun}_target_enc'].fillna(genel_ortalama, inplace=True)

    # 3. Target Encoding'li yeni matrisle eğitim
    model_yeni = LGBMRegressor(random_state=42, n_jobs=-1)
    model_yeni.fit(X_train_yeni, y_train_fold)
    tahmin_yeni = model_yeni.predict(X_val_yeni)
    rmse_te_matris.append(np.sqrt(mean_squared_error(y_val_fold, tahmin_yeni)))

print("-" * 40)
eski_ort = np.mean(rmse_eski_matris)
yeni_ort = np.mean(rmse_te_matris)

print(f"Orijinal Matris ile Ortalama RMSE : {eski_ort:.4f}")
print(f"Target Encoded Matris ile RMSE    : {yeni_ort:.4f}")

fark = eski_ort - yeni_ort
if fark > 0:
    print(f"\nSONUÇ: BAŞARILI! Skor {fark:.5f} iyileşti. Bu yeni özellikleri ana matrisimize kalıcı olarak ekleyelim.")
else:
    print(f"\nSONUÇ: BAŞARISIZ. Skor kötüleşti. Target Encoding sızıntı veya gürültü yarattı, eklemeden devam ediyoruz.")

----------------------------------------
Orijinal Matris ile Ortalama RMSE : 1.2341
Target Encoded Matris ile RMSE    : 1.2321

SONUÇ: BAŞARILI! Skor 0.00198 iyileşti. Bu yeni özellikleri ana matrisimize kalıcı olarak ekleyelim.


In [31]:
# Sızıntıyı (Leakage) önlemek için Out-of-Fold (OOF) stratejisi ile matrisi güncelliyoruz
X['meslek_target_enc'] = np.nan
X['ulke_target_enc'] = np.nan

kf_kalici = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf_kalici.split(X):
    for sutun in ['meslek', 'ulke']:
        # Sadece diğer 4 parçadaki ortalamayı öğren
        ortalama = y.iloc[train_idx].groupby(train.iloc[train_idx][sutun]).mean()
        # O 1 parçanın (validasyon) karşısına bu ortalamayı yaz
        X.loc[X.index[val_idx], f'{sutun}_target_enc'] = train.iloc[val_idx][sutun].map(ortalama)

# Test seti için doğrudan tüm eğitim setinin ortalamasını haritalıyoruz
for sutun in ['meslek', 'ulke']:
    genel_ortalama_train = y.groupby(train[sutun]).mean()
    X_test[f'{sutun}_target_enc'] = test[sutun].map(genel_ortalama_train)

# Eşleşmeyen (NaN) değerler kalırsa genel hedef ortalamasıyla dolduruyoruz
genel_y_ort = y.mean()
X.fillna(genel_y_ort, inplace=True)
X_test.fillna(genel_y_ort, inplace=True)

print(f"Target Encoding özellikleri ana matrise sızıntısız eklendi! Güncel X boyutu: {X.shape}")

Target Encoding özellikleri ana matrise sızıntısız eklendi! Güncel X boyutu: (56000, 51)


---
## 4. Deney 3: K-Means Kümeleme (Clustering) Sinyali
**Hipotez:** Sürekli değişkenler (uyku süresi, stres, çalışma saati vb.) kullanılarak hastalar/kişiler K-Means ile kümelenecek ve çıkan "Küme ID"si modele yeni bir kategorik özellik olarak verilecektir.

In [32]:
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler

# # X matrisinin içinde olduğu teyit edilen kolonlar:
# kume_kolonlari = [
#     'bilissel_yuk_endeksi', 
#     'uyku_hijyeni_ihlali', 
#     'derin_uyku_yuzdesi', 
#     'gecelik_uyanma_sayisi', 
#     'yas'
# ]

# # Matrisi kopyalayalım
# X_kume_zengin = X.copy()

# # 1. Ölçeklendirme (Doğrudan X üzerinden çekiyoruz)
# scaler = StandardScaler()
# # X içindeki ilgili kolonları alıp ölçeklendiriyoruz
# X_scaled = scaler.fit_transform(X[kume_kolonlari].fillna(0))

# # 2. İnsanları 5 "Yaşam ve Performans Profili"ne ayırıyoruz
# kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
# X_kume_zengin['insan_profili_kume'] = kmeans.fit_predict(X_scaled)

# # Kategorik olarak işaretleyelim (LightGBM için kritik)
# X_kume_zengin['insan_profili_kume'] = X_kume_zengin['insan_profili_kume'].astype('category')

# # 3. Model testi (Cross-Validation)
# rmse_kume_zengin = []
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# for train_index, val_index in kf.split(X_kume_zengin):
#     X_train_kz, X_val_kz = X_kume_zengin.iloc[train_index], X_kume_zengin.iloc[val_index]
#     y_train_kz, y_val_kz = y.iloc[train_index], y.iloc[val_index]
    
#     # Parametresiz LightGBM ile test ediyoruz
#     model_kz = LGBMRegressor(random_state=42, n_jobs=-1)
#     model_kz.fit(X_train_kz, y_train_kz)
#     tahmin_kz = model_kz.predict(X_val_kz)
#     rmse_kume_zengin.append(np.sqrt(mean_squared_error(y_val_kz, tahmin_kz)))

# yeni_skor_kz = np.mean(rmse_kume_zengin)

# print("-" * 40)
# # yeni_ort: Bir önceki (Target Encoding) adımdan gelen skorun olduğunu varsayıyorum
# print(f"Target Encoding Sonrası RMSE: {yeni_ort:.4f}") 
# print(f"Zengin K-Means Eklenmiş RMSE: {yeni_skor_kz:.4f}")

# fark_kz = yeni_ort - yeni_skor_kz
# if fark_kz > 0:
#     print(f"\nSONUÇ: BAŞARILI! {fark_kz:.5f} iyileşme var. Bu 'zengin' profili kalıcı yapıyoruz.")
# else:
#     print(f"\nSONUÇ: BAŞARISIZ. Gürültü arttı, kümeleme sütununu eklemiyoruz.")

---
## 5. Deney 4: Ensemble (Model Blending)
**Strateji:** Tek bir modele güvenmek yerine, üç farklı dev algoritmanın (LGBM, XGBoost, CatBoost) tahminlerini birleştiriyoruz. Bu yöntem varyansı düşürür ve liderlik tablosunda en sağlam sıçramayı yaptırır.

In [33]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# 1. Modelleri tanımlıyoruz (Şimdilik makul genel parametrelerle)
# Not: CatBoost kategorik verileri çok sever ama biz zaten OHE yaptık, sorun yok.

lgbm_model = LGBMRegressor(
    learning_rate=0.079, max_depth=10, num_leaves=42, 
    subsample=0.71, colsample_bytree=0.70, min_child_samples=44,
    random_state=42, n_jobs=-1, verbose=-1
)

xgb_model = XGBRegressor(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, 
    tree_method='hist', random_state=42, n_jobs=-1
)

cat_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, bootstrap_type='Bayesian',
    random_state=42, verbose=0
)

# 2. Cross-Validation ile Ensemble testi
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_rmse_skorlari = []

print("Ensemble eğitimi başlıyor (Bu işlem 1-2 dk sürebilir)...")

for train_index, val_index in kf.split(X):
    X_train_e, X_val_e = X.iloc[train_index], X.iloc[val_index]
    y_train_e, y_val_e = y.iloc[train_index], y.iloc[val_index]
    
    # Modelleri eğit
    lgbm_model.fit(X_train_e, y_train_e)
    xgb_model.fit(X_train_e, y_train_e)
    cat_model.fit(X_train_e, y_train_e)
    
    # Tahminleri al
    preds_lgbm = lgbm_model.predict(X_val_e)
    preds_xgb = xgb_model.predict(X_val_e)
    preds_cat = cat_model.predict(X_val_e)
    
    # --- BLENDING (Ağırlıklı Ortalama) ---
    # Başlangıç ağırlıkları: %40 LGBM, %35 XGB, %25 Cat
    final_preds = (preds_lgbm * 0.40) + (preds_xgb * 0.30) + (preds_cat * 0.30)
    
    hata = np.sqrt(mean_squared_error(y_val_e, final_preds))
    ensemble_rmse_skorlari.append(hata)

print("-" * 40)
print(f"LGBM Tek Başına (Eski): 1.2321")
print(f"ENSEMBLE Ortalama RMSE : {np.mean(ensemble_rmse_skorlari):.4f}")

Ensemble eğitimi başlıyor (Bu işlem 1-2 dk sürebilir)...
----------------------------------------
LGBM Tek Başına (Eski): 1.2321
ENSEMBLE Ortalama RMSE : 1.2241


---
## 6. Nihai Ensemble Eğitimi ve Submission
**Metot:** En düşük RMSE'yi veren ağırlıklarla (LGBM: 0.40, XGB: 0.35, CAT: 0.25) tüm veri seti üzerinde modeller eğitilir ve test seti tahminleri birleştirilir.

In [34]:
# 1. Modelleri TÜM veriyle (X ve y) eğitiyoruz
print("Modeller son kez eğitiliyor...")
lgbm_model.fit(X, y)
xgb_model.fit(X, y)
cat_model.fit(X, y)

# 2. Test seti (X_test) tahminlerini alıyoruz
test_preds_lgbm = lgbm_model.predict(X_test)
test_preds_xgb = xgb_model.predict(X_test)
test_preds_cat = cat_model.predict(X_test)

# 3. Belirlediğimiz başarılı ağırlıklarla harmanlıyoruz
final_test_preds = (test_preds_lgbm * 0.40) + (test_preds_xgb * 0.35) + (test_preds_cat * 0.25)

# 4. Submission dosyasını oluşturma
submission_ensemble = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': final_test_preds
})

submission_ensemble.to_csv('final_ensemble_submission.csv', index=False)
print("Tebrikler! 'final_ensemble_submission.csv' hazır.")

Modeller son kez eğitiliyor...
Tebrikler! 'final_ensemble_submission.csv' hazır.


In [35]:
import optuna

def blending_objective(trial):
    # Üç ağırlığın toplamının 1 olması gerekir
    w1 = trial.suggest_float('w_lgbm', 0.0, 1.0)
    w2 = trial.suggest_float('w_xgb', 0.0, 1.0)
    w3 = trial.suggest_float('w_cat', 0.0, 1.0)
    
    # Ağırlıkları normalize et (toplamı 1 yap)
    total = w1 + w2 + w3
    w1, w2, w3 = w1/total, w2/total, w3/total
    
    # Daha önce aldığımız Cross-Validation tahminlerini kullanıyoruz
    # (Not: Bu kodun çalışması için CV döngüsünde 'preds' değerlerini saklamış olmalıyız)
    # Şimdilik mantığı anlaman için basitleştirilmiş hali:
    
    combined_preds = (preds_lgbm * w1) + (preds_xgb * w2) + (preds_cat * w3)
    rmse = np.sqrt(mean_squared_error(y_val_e, combined_preds))
    return rmse

# Bu çalışma sadece birkaç saniye sürer çünkü model eğitmez, sadece sayı çarpar.

In [36]:
# def xgb_objective(trial):
#     param = {
#         'n_estimators': 1000,
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
#         'tree_method': 'hist',
#         'random_state': 42,
#         'n_jobs': -1
#     }
#     kf = KFold(n_splits=5, shuffle=True, random_state=42)
#     rmses = []
#     for train_idx, val_idx in kf.split(X):
#         model = XGBRegressor(**param)
#         model.fit(X.iloc[train_idx], y.iloc[train_idx])
#         rmses.append(np.sqrt(mean_squared_error(y.iloc[val_idx], model.predict(X.iloc[val_idx]))))
#     return np.mean(rmses)

# study_xgb = optuna.create_study(direction='minimize')
# study_xgb.optimize(xgb_objective, n_trials=20) # Vaktin varsa n_trials=50 yapabilirsin

In [37]:
# def cat_objective(trial):
#     param = {
#         'iterations': 1000,
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'depth': trial.suggest_int('depth', 4, 10),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
#         'bootstrap_type': 'Bayesian',
#         'random_state': 42,
#         'verbose': 0
#     }
#     kf = KFold(n_splits=5, shuffle=True, random_state=42)
#     rmses = []
#     for train_idx, val_idx in kf.split(X):
#         model = CatBoostRegressor(**param)
#         model.fit(X.iloc[train_idx], y.iloc[train_idx])
#         rmses.append(np.sqrt(mean_squared_error(y.iloc[val_idx], model.predict(X.iloc[val_idx]))))
#     return np.mean(rmses)

# study_cat = optuna.create_study(direction='minimize')
# study_cat.optimize(cat_objective, n_trials=20)

---
## 7. Büyük Final: Optimized Stacking Ensemble
**Strateji:** Tek başına 1.220 skoru atan CatBoost, 1.225 atan XGBoost ve 1.232 atan LightGBM modelleri, bir 'Meta-Model' (RidgeCV) altında birleştirilir. Bu yapı, her modelin hata yaptığı noktaları diğerleriyle kapatmasını sağlar.

In [38]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

# 1. Optuna'dan gelen "En İyi" modelleri tanımlıyoruz
best_lgbm = LGBMRegressor(
    learning_rate=0.079675, max_depth=10, num_leaves=42, 
    subsample=0.7175, colsample_bytree=0.7017, min_child_samples=44,
    random_state=42, n_jobs=-1, verbose=-1
)

best_xgb = XGBRegressor(
    learning_rate=0.010242, max_depth=5, 
    subsample=0.5103, colsample_bytree=0.8068,
    n_estimators=1000, tree_method='hist', random_state=42, n_jobs=-1
)

best_cat = CatBoostRegressor(
    learning_rate=0.045364, depth=5, l2_leaf_reg=5.7870,
    iterations=1000, bootstrap_type='Bayesian',
    random_state=42, verbose=0
)

# 2. Stacking Mimarisini kuruyoruz
# cv=5 parametresi, meta-modelin eğitimi için modellerin 5-fold tahmin üretmesini sağlar
estimators = [
    ('lgbm', best_lgbm),
    ('xgb', best_xgb),
    ('cat', best_cat)
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(), # Meta-model: Ağırlıkları akıllıca belirleyen Ridge
    cv=5,
    n_jobs=-1,
    passthrough=False # Sadece model tahminlerini kullan, orijinal X'i meta-modele sokma
)

print("Zirve modeli (Stacking) eğitiliyor... Bu işlem birkaç dakika sürebilir.")
stack_model.fit(X, y)

# 3. Test Seti Tahmini ve Submission
print("Tahminler üretiliyor...")
stack_preds = stack_model.predict(X_test)

submission_final = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': stack_preds
})

submission_final.to_csv('ultimate_stacking_submission.csv', index=False)
print("İşlem tamam! 'submission1.csv' dosyasını Kaggle'a göndermeye hazırsın.")

Zirve modeli (Stacking) eğitiliyor... Bu işlem birkaç dakika sürebilir.


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_2ff1fd19811840d5a1a2d4093bcfbf23_e0391ddd34b142eea66c7a9e26e5e1ce for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-8357-sl90vfjn for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/py

Tahminler üretiliyor...
İşlem tamam! 'submission1.csv' dosyasını Kaggle'a göndermeye hazırsın.


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-8357-dgv9148i for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-8357-9jpagavd for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for 

İlk Submission burda gönderildi


Public Score: 1.20837


In [39]:
from sklearn.model_selection import cross_val_predict
import optuna

# 1. Her modelin OOF (Out-of-Fold) tahminlerini alalım
# Bu, modellerin eğitim setindeki 'hiç görmedikleri' satırlara yaptığı tahminlerdir.
print("LGBM OOF tahminleri alınıyor...")
oof_lgbm = cross_val_predict(best_lgbm, X, y, cv=5, n_jobs=-1)

print("XGB OOF tahminleri alınıyor...")
oof_xgb = cross_val_predict(best_xgb, X, y, cv=5, n_jobs=-1)

print("CatBoost OOF tahminleri alınıyor...")
oof_cat = cross_val_predict(best_cat, X, y, cv=5, n_jobs=-1)

# 2. Ağırlıkları optimize eden Optuna fonksiyonu
def weight_objective(trial):
    # Üç model için ağırlıklar (0 ile 1 arası)
    w_lgbm = trial.suggest_float('w_lgbm', 0.0, 1.0)
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cat = trial.suggest_float('w_cat', 0.0, 1.0)
    
    # Ağırlıkları toplamı 1 olacak şekilde normalize et
    total = w_lgbm + w_xgb + w_cat
    w1, w2, w3 = w_lgbm/total, w_xgb/total, w_cat/total
    
    # Ağırlıklı ortalamayı hesapla
    combined_oof_preds = (oof_lgbm * w1) + (oof_xgb * w2) + (oof_cat * w3)
    
    # Local RMSE hesapla
    rmse = np.sqrt(mean_squared_error(y, combined_oof_preds))
    return rmse

# 3. Optuna'yı çalıştır (100 deneme yeterli olacaktır)
study_weights = optuna.create_study(direction='minimize')
study_weights.optimize(weight_objective, n_trials=100)

print("-" * 40)
print(f"EN İYİ LOCAL RMSE: {study_weights.best_value:.6f}")
print("EN İYİ AĞIRLIKLAR:", study_weights.best_params)

# En iyi ağırlıkları normalize edip alalım
best_w = study_weights.best_params
total_w = best_w['w_lgbm'] + best_w['w_xgb'] + best_w['w_cat']
w1, w2, w3 = best_w['w_lgbm']/total_w, best_w['w_xgb']/total_w, best_w['w_cat']/total_w

print(f"\nFinal Ağırlıklar: LGBM: {w1:.4f}, XGB: {w2:.4f}, CAT: {w3:.4f}")

LGBM OOF tahminleri alınıyor...


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_a13363535f57401fbe91db371797d6e6_42a35646832b4663920076fd1eacb927 for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_2ff1fd19811840d5a1a2d4093bcfbf23_a93ba53cd6594639a83d28f4848052ba for automatic cleanup: unkno

XGB OOF tahminleri alınıyor...


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_c8be370bef5741fc8a46f9ba37f285b6_34006d3eca11464b9eafa43159d37766 for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_2ff1fd19811840d5a1a2d4093bcfbf23_ee8c215ffbb0440eacee2a661be48584 for automatic cleanup: unkno

CatBoost OOF tahminleri alınıyor...


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_29455174c29b4424a9020e241542e11c_3ea7cd024ad04c88a9d6c576a4337663 for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/s0/tf_n875s3ps06ts4061yxy7c0000gn/T/joblib_memmapping_folder_8357_2ff1fd19811840d5a1a2d4093bcfbf23_c86b0419177e4740a35e3894741b5e24 for automatic cleanup: unkno

----------------------------------------
EN İYİ LOCAL RMSE: 1.219719
EN İYİ AĞIRLIKLAR: {'w_lgbm': 0.19249526762787744, 'w_xgb': 0.0008653798590040895, 'w_cat': 0.9561426333441857}

Final Ağırlıklar: LGBM: 0.1675, XGB: 0.0008, CAT: 0.8318


## ADIM 8: Pseudo-Labeling Operasyonu
**Hipotez:** Test setindeki verilerin dağılımını modele öğretmek için en iyi tahminlerimizi 'gerçek' kabul edip eğitim setini genişletiyoruz.

In [40]:
# 1. En iyi tahminlerini içeren submission dosyasını oku
# Dosya adını senin en son oluşturduğun (1.208 skorlu olan) isimle değiştir
best_submission = pd.read_csv('submission.csv')

# 2. Test verisiyle tahminleri birleştirerek 'sahte' bir eğitim seti kuralım
X_pseudo = X_test.copy()
y_pseudo = best_submission['bilissel_performans_skoru']

# 3. Orijinal X ve y ile bu sahte verileri birleştiriyoruz
X_final_augmented = pd.concat([X, X_pseudo], axis=0).reset_index(drop=True)
y_final_augmented = pd.concat([y, y_pseudo], axis=0).reset_index(drop=True)

print(f"Orijinal Satır Sayısı: {len(X)}")
print(f"Yeni (Genişletilmiş) Satır Sayısı: {len(X_final_augmented)}")

# 4. Şimdi en iyi modelimizi (CatBoost ağırlıklı olanı veya Stacking'i) 
# bu devasa veriyle tekrar eğitiyoruz. 
# CatBoost bu kadar çok veride harikalar yaratır.

print("Pseudo-Labeling ile model eğitiliyor...")
# CatBoost'u genişletilmiş veriyle eğitelim
final_cat_pseudo = CatBoostRegressor(
    learning_rate=0.045364, depth=5, l2_leaf_reg=5.7870,
    iterations=1500, # Veri arttığı için iterasyonu biraz artırabiliriz
    bootstrap_type='Bayesian',
    random_state=42, verbose=0
)

final_cat_pseudo.fit(X_final_augmented, y_final_augmented)

# 5. Sadece orijinal X_test için nihai tahminleri yapalım
final_pseudo_preds = final_cat_pseudo.predict(X_test)

# Submission
submission_pseudo = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': final_pseudo_preds
})

submission_pseudo.to_csv('pseudo_labeling_submission.csv', index=False)
print("Şampiyonluk adayı dosya hazır: 'pseudo_labeling_submission.csv'")

Orijinal Satır Sayısı: 56000
Yeni (Genişletilmiş) Satır Sayısı: 80000
Pseudo-Labeling ile model eğitiliyor...
Şampiyonluk adayı dosya hazır: 'pseudo_labeling_submission.csv'


### Pseudo-Labeling Doğrulama (Validation)
**Metot:** Orijinal eğitim verisinin %10'u "dokunulmaz" olarak ayrılır. Kalan %90 ve tahmin edilmiş test verisi birleştirilerek model eğitilir. Başarı, dokunulmaz %10'luk veri üzerinden ölçülür.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Orijinal train verisini %90 eğitim, %10 doğrulama (validation) olarak bölüyoruz.
X_train_part, X_val_clean, y_train_part, y_val_clean = train_test_split(
    X, y, test_size=0.10, random_state=42
)

# 2. Test verisini ve senin en iyi tahminlerini alalım
best_submission = pd.read_csv('ultimate_stacking_submission.csv')
X_pseudo = X_test.copy()
y_pseudo = best_submission['bilissel_performans_skoru']

# 3. Eğitim havuzunu oluştur: %90 train + TÜM test verisi
X_augmented_train = pd.concat([X_train_part, X_pseudo], axis=0)
y_augmented_train = pd.concat([y_train_part, y_pseudo], axis=0)

# 4. Modeli bu havuzda eğit
model_pseudo_val = CatBoostRegressor(
    learning_rate=0.045364, depth=5, l2_leaf_reg=5.7870,
    iterations=1500, bootstrap_type='Bayesian',
    random_state=42, verbose=0
)

print("Genişletilmiş veriyle eğitim başlıyor...")
model_pseudo_val.fit(X_augmented_train, y_augmented_train)

# 5. TEST: Modelin hiç görmediği o saf %10'luk orijinal veride skoru ölç
val_preds = model_pseudo_val.predict(X_val_clean)
local_pseudo_rmse = np.sqrt(mean_squared_error(y_val_clean, val_preds))

print("-" * 40)
print(f"HİÇ GÖRÜLMEMİŞ VERİDEKİ LOCAL RMSE: {local_pseudo_rmse:.6f}")

Genişletilmiş veriyle eğitim başlıyor...
----------------------------------------
HİÇ GÖRÜLMEMİŞ VERİDEKİ LOCAL RMSE: 1.217093


Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-8357-puebqga5 for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-8357-vgpd8xfv for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/resource_tracker.py", line 295, in main
    raise ValueError(
        f'Cannot register {name} for 

In [42]:
import numpy as np

# 1. Havuzu tekrar kuralım (Tüm train + tahmin edilmiş test)
X_final_augmented = pd.concat([X, X_test], axis=0).reset_index(drop=True)
y_final_augmented = pd.concat([y, best_submission['bilissel_performans_skoru']], axis=0).reset_index(drop=True)

# 2. Seed Averaging (Farklı bakış açıları için 5 model)
seeds = [42, 1903, 2026, 7, 88]
bagging_preds = []

print("Zirveye giden final modelleri eğitiliyor (Bagging)...")

for seed in seeds:
    print(f"Eğitiliyor: Seed {seed}")
    model = CatBoostRegressor(
        learning_rate=0.045364, 
        depth=5, 
        l2_leaf_reg=5.7870,
        iterations=1500, 
        bootstrap_type='Bayesian',
        random_state=seed, # Her seferinde farklı bir başlangıç
        verbose=0
    )
    model.fit(X_final_augmented, y_final_augmented)
    bagging_preds.append(model.predict(X_test))

# 3. Modellerin ortalamasını al (Final Sinerji)
final_submission_preds = np.mean(bagging_preds, axis=0)

# 4. Dosyayı oluştur
submission_champ = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': final_submission_preds
})

submission_champ.to_csv('final_pseudo_bagging.csv', index=False)
print("BÜYÜK FİNAL DOSYASI HAZIR: final_pseudo_bagging.csv")

Zirveye giden final modelleri eğitiliyor (Bagging)...
Eğitiliyor: Seed 42
Eğitiliyor: Seed 1903
Eğitiliyor: Seed 2026
Eğitiliyor: Seed 7
Eğitiliyor: Seed 88
BÜYÜK FİNAL DOSYASI HAZIR: final_pseudo_bagging.csv


In [43]:

# 1. Orijinal veriden "kurban" %10'u tekrar ayırıyoruz
X_train_part, X_val_clean, y_train_part, y_val_clean = train_test_split(
    X, y, test_size=0.10, random_state=42
)

# 2. Pseudo-labels (Senin 1.208'lik tahminlerin)
best_submission = pd.read_csv('ultimate_stacking_submission.csv')
X_pseudo = X_test.copy()
y_pseudo = best_submission['bilissel_performans_skoru']

# 3. Eğitim havuzu: %90 train + TÜM test verisi
X_augmented_train = pd.concat([X_train_part, X_pseudo], axis=0)
y_augmented_train = pd.concat([y_train_part, y_pseudo], axis=0)

# 4. Bagging Testi (5 farklı seed ile ortalama alma)
seeds = [42, 1903, 2026, 7, 88]
val_bagging_preds = []

print("Bagging etkisi ölçülüyor...")

for seed in seeds:
    model_bag = CatBoostRegressor(
        learning_rate=0.045364, depth=5, l2_leaf_reg=5.7870,
        iterations=1500, bootstrap_type='Bayesian',
        random_state=seed, verbose=0
    )
    model_bag.fit(X_augmented_train, y_augmented_train)
    val_bagging_preds.append(model_bag.predict(X_val_clean))

# 5. Ortalamayı al ve skoru ölç
final_val_preds = np.mean(val_bagging_preds, axis=0)
local_bagging_rmse = np.sqrt(mean_squared_error(y_val_clean, final_val_preds))

print("-" * 40)
print(f"Tek Model Pseudo RMSE: 1.21709")
print(f"Bagging + Pseudo RMSE: {local_bagging_rmse:.6f}")

fark = 1.21709 - local_bagging_rmse
if fark > 0:
    print(f"GELİŞME: Bagging skoru {fark:.6f} kadar daha iyileştirdi!")
else:
    print("GELİŞME: Bagging bu sefer fark yaratmadı.")

Bagging etkisi ölçülüyor...
----------------------------------------
Tek Model Pseudo RMSE: 1.21709
Bagging + Pseudo RMSE: 1.216844
GELİŞME: Bagging skoru 0.000246 kadar daha iyileştirdi!


In [ ]:
# TÜM veriyi (Orijinal + Pseudo) kullanarak final tahminini üretelim
print("Final dosyası için tüm veriyle Bagging süreci başlıyor...")

# Havuzu tekrar full kapasite kuralım
X_final_augmented = pd.concat([X, X_test], axis=0).reset_index(drop=True)
y_final_augmented = pd.concat([y, best_submission['bilissel_performans_skoru']], axis=0).reset_index(drop=True)

final_seeds = [42, 1903, 2026, 7, 88, 10, 55] # Seed sayısını 7'ye çıkardım, daha stabil olsun
final_bagging_preds = []

for seed in final_seeds:
    model = CatBoostRegressor(
        learning_rate=0.045364, 
        depth=5, 
        l2_leaf_reg=5.7870,
        iterations=1500, 
        bootstrap_type='Bayesian',
        random_state=seed,
        verbose=0
    )
    model.fit(X_final_augmented, y_final_augmented)
    final_bagging_preds.append(model.predict(X_test))

# 7 modelin ortalaması (Varyansı minimize eder)
ultimate_preds = np.mean(final_bagging_preds, axis=0)

submission_champ = pd.DataFrame({
    'id': test_id,
    'bilissel_performans_skoru': ultimate_preds
})

submission_champ.to_csv('absolute_final_pseudo_bagging.csv', index=False)
print("DOSYA HAZIR: 'absolute_final_pseudo_bagging.csv'. Bu bizim şu anki zirve noktamız.")

Efsanevi final dosyası için tüm veriyle Bagging süreci başlıyor...
DOSYA HAZIR: 'absolute_final_pseudo_bagging.csv'. Bu bizim şu anki zirve noktamız.
